# SkillMem Run 4: can the agent solve the 3 weak tasks? (3 tasks x conditions A, C)
**Settings:** GPU **T4 x2**, Internet **on**. **Secrets:** `HF_TOKEN` + `CLAUDE_CODE_OAUTH_TOKEN`.
**Run:** *Save Version -> Save & Run All (Commit)*. Stops cleanly after 8 h. Results: `run4_tasks_results.tar.gz` in Output. It can resume only if HF has free space.

Motion and detection hints were written by a separate Claude Code session after reading Run 3's failures, so report this run as hint-assisted.
Held-out seeds are 6-10. This run is not a memory comparison.

In [ ]:
# 1. Secrets, GPU check, settings
import os, subprocess, json, time, pathlib, shutil
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
os.environ["HF_TOKEN"] = sec.get_secret("HF_TOKEN")
try:
    os.environ["CLAUDE_CODE_OAUTH_TOKEN"] = sec.get_secret("CLAUDE_CODE_OAUTH_TOKEN")
except Exception:
    raise SystemExit("Run 1 needs the CLAUDE_CODE_OAUTH_TOKEN secret (Add-ons -> Secrets, tick it)")
gpus = subprocess.check_output(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total",
                                "--format=csv,noheader"]).decode().strip().splitlines()
print(gpus)
assert all("T4" in g for g in gpus), "Choose accelerator 'GPU T4 x2'. P100 is too old for ASPIRE's torch 2.9 build."
NGPU = len(gpus)
PERCEPTION = "sam3"    # "sam3" = ASPIRE's original (access approved). Backup: "gsam2" = open Grounded-SAM 2 stand-in.
SETTLE_STEPS = 300     # ASPIRE waits 120 sim steps for the arm to reach a move's final pose; big wrist turns
                       # need ~133 at the arm's ~0.15 rad/s, so they time out. 300 for every task (documented deviation).
RUN_SMOKE = False      # ASPIRE's structural smoke matrix (cell 8) adds 10-20 min; doctor in cell 7 already covers it
RUN, REPO, COMMIT = "run4_tasks", "/root/aspire", "fa00f0ef2bc1d91d594b9aa502c22133c9f538f7"
BUDGET_H = 8.0  # stop cleanly after this many hours (10 h of GPU quota were left)
os.environ["SKILLMEM_DEADLINE"] = str(time.time() + BUDGET_H * 3600)
OUT = f"/kaggle/working/{RUN}"; os.makedirs(OUT, exist_ok=True)
os.environ.update(
    CAP_HARNESS_PROFILE="t4", SAM3_DTYPE="float16",           # T4 has no bf16; SAM3 patched to fp16 (cell 4)
    LIBERO_CONFIG_PATH=f"{REPO}/.libero", TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD="1",
    MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl", EGL_PLATFORM="device", MUJOCO_EGL_DEVICE_ID="0",
    UV_LINK_MODE="copy", PATH=f"/root/.local/bin:{os.environ['PATH']}",
    MPLBACKEND="Agg")   # Kaggle sets an inline backend that ASPIRE's own venv doesn't have
!df -h / /kaggle/working | tail -2

In [ ]:
%%writefile /kaggle/working/ckpt.py
"""Checkpoints to a private HF dataset repo <hf-user>/skillmem-ckpt, so a dead Kaggle session loses at most ~20 min.
save(run, paths): tar.gz the paths (minus keyframe PNGs) -> <run>/latest.tar.gz; small=True uploads only state.json+programs.
restore(run): download <run>/latest.tar.gz and unpack it in place. Returns False if none exists.
State: <OUT>/state.json lists finished steps, so a rerun skips them."""
import os, tarfile, time, threading, json
from huggingface_hub import HfApi, hf_hub_download

_repo = None
def api():
    return HfApi(token=os.environ["HF_TOKEN"])

def repo():
    global _repo
    if _repo is None:
        a = api(); _repo = f"{a.whoami()['name']}/skillmem-ckpt"
        a.create_repo(_repo, repo_type="dataset", private=True, exist_ok=True)
    return _repo

_last_hourly = 0
_lock = threading.Lock()
SKIP = ("/media/keyframes/", "/media/overlays/")   # big PNG folders; the frames the agent needs are copied elsewhere
def _filt(ti):
    return None if any(k in "/" + ti.name for k in SKIP) else ti
def _small(ti):  # state.json + programs only (a few KB): saved after every Claude call
    n = "/" + ti.name
    return ti if (ti.isdir() or n.endswith("/state.json") or "/programs/" in n) else None
def _save(run, paths, note="", small=False):
    """Full save: everything except keyframe PNGs -> <run>/latest.tar.gz. Always also uploads <run>/state.tar.gz."""
    global _last_hourly
    with _lock:
        a = api(); msg = f"{run} {time.strftime('%Y-%m-%d %H:%M')} {note}"
        jobs = [("state", _small)]  # Run 4: never upload the full tar (every version stays in LFS history)
        for name, f in jobs:
            tar = f"/tmp/{run}-{name}.tar.gz"
            with tarfile.open(tar, "w:gz") as t:
                for p in paths:
                    if os.path.exists(p):
                        t.add(p, arcname=p.lstrip("/"), filter=f)
            a.upload_file(path_or_fileobj=tar, path_in_repo=f"{run}/{name}.tar.gz", repo_id=repo(), repo_type="dataset", commit_message=msg)
            if name == "state" and time.time() - _last_hourly > 3600:
                a.upload_file(path_or_fileobj=tar, path_in_repo=f"{run}/history/state-{time.strftime('%Y%m%d-%H%M')}.tar.gz",
                              repo_id=repo(), repo_type="dataset", commit_message=msg + " (hourly)")
                _last_hourly = time.time()
            print(f"[ckpt] saved {run}/{name} ({os.path.getsize(tar) / 1e6:.1f} MB) {note}", flush=True)

def save(run, paths, note="", small=False):
    try:
        _save(run, paths, note, small)
        return True
    except Exception as e:
        print(f"[ckpt] HF save failed, continuing (results stay in /kaggle/working): {str(e)[:250]}", flush=True)
        return False

def pack(run, paths):
    out = f"/kaggle/working/{run}_results.tar.gz"
    try:
        with _lock, tarfile.open(out + ".part", "w:gz", compresslevel=1) as t:
            for p in paths:
                if os.path.exists(p):
                    t.add(p, arcname=p.lstrip("/"), filter=_filt)
        os.replace(out + ".part", out)
    except Exception as e:
        print(f"[ckpt] pack failed, continuing: {e}", flush=True)
        return
    print(f"[ckpt] packed {out} ({os.path.getsize(out) / 1e6:.1f} MB): download it from the Output tab", flush=True)

def restore(run):
    ok = False
    for name in ("latest", "state"):   # state.tar.gz is always the newest, so it is unpacked last
        try:
            f = hf_hub_download(repo(), f"{run}/{name}.tar.gz", repo_type="dataset", token=os.environ["HF_TOKEN"])
        except Exception as e:
            print(f"[ckpt] no {name} checkpoint for {run} ({type(e).__name__})"); continue
        with tarfile.open(f) as t:
            t.extractall("/")
        print(f"[ckpt] restored {run}/{name}"); ok = True
    return ok

def autosave(run, paths, every_min=20):
    def loop():
        while True:
            time.sleep(every_min * 60)
            try: save(run, paths, "auto")
            except Exception as e: print("[ckpt] autosave failed:", e)
    threading.Thread(target=loop, daemon=True).start()

class State:
    def __init__(self, out):
        self.path = f"{out}/state.json"
        self.d = json.load(open(self.path)) if os.path.exists(self.path) else {"done": {}}
    def done(self, step): return step in self.d["done"]
    def mark(self, step, info=None):
        self.d["done"][step] = info if info is not None else True
        json.dump(self.d, open(self.path, "w"), indent=1)

In [ ]:
# 2. Restore the last checkpoint (if any) and start autosave every 20 min
import sys; sys.path.insert(0, "/kaggle/working")
import importlib, ckpt; importlib.reload(ckpt)
ckpt.restore(RUN)
CK_PATHS = [OUT]
state = ckpt.State(OUT)
print("already done:", list(state.d["done"]))
ckpt.autosave(RUN, CK_PATHS, every_min=30)

In [ ]:
%%writefile /kaggle/working/gsam2_backend.py
"""Grounded-SAM 2: an openly downloadable stand-in for SAM3 inside ASPIRE (no gated weights).

  text prompt -> Grounding DINO boxes -> SAM 2.1 masks inside each box   (replaces SAM3 text segmentation)
  pixel (x,y) -> SAM 2.1 multimask output                                (replaces SAM3 point segmentation)

Both models are Apache-2.0 and download from Hugging Face without an access request.
Interface matches what ASPIRE's SAM3 service needs:
  segment_text(image, prompt)  -> masks (N,H,W) bool, boxes (N,4) xyxy pixels, scores (N,)   sorted by score
  segment_point(image, (x, y)) -> masks (N,H,W) bool, scores (N,)                             sorted by score
"""
import os

import numpy as np
import torch
from transformers import AutoModelForZeroShotObjectDetection, AutoProcessor, Sam2Model, Sam2Processor

DINO_ID = os.environ.get("GSAM_DINO", "IDEA-Research/grounding-dino-base")
SAM_ID = os.environ.get("GSAM_SAM", "facebook/sam2.1-hiera-large")
BOX_THRESHOLD = float(os.environ.get("GSAM_BOX_THRESHOLD", "0.20"))  # low: ASPIRE ran SAM3 with threshold 0
TEXT_THRESHOLD = float(os.environ.get("GSAM_TEXT_THRESHOLD", "0.20"))
MAX_RESULTS = 5


class GroundedSAM2:
    def __init__(self, device="cuda"):
        self.device = device
        self.dino_proc = AutoProcessor.from_pretrained(DINO_ID)
        self.dino = AutoModelForZeroShotObjectDetection.from_pretrained(DINO_ID).to(device).eval()
        self.sam_proc = Sam2Processor.from_pretrained(SAM_ID)
        self.sam = Sam2Model.from_pretrained(SAM_ID).to(device).eval()

    @torch.inference_mode()
    def _boxes(self, image, prompt):
        text = prompt.strip().lower()
        if not text.endswith("."):
            text += "."  # Grounding DINO expects lowercase phrases ending with a period
        inputs = self.dino_proc(images=image, text=text, return_tensors="pt").to(self.device)
        out = self.dino(**inputs)
        res = self.dino_proc.post_process_grounded_object_detection(
            out, inputs.input_ids, threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD,
            target_sizes=[image.size[::-1]])[0]
        boxes, scores = res["boxes"].float().cpu().numpy(), res["scores"].float().cpu().numpy()
        order = np.argsort(-scores)[:MAX_RESULTS]
        return boxes[order], scores[order]

    @torch.inference_mode()
    def segment_text(self, image, prompt):
        w, h = image.size
        boxes, scores = self._boxes(image, prompt)
        if len(boxes) == 0:
            return np.zeros((0, h, w), bool), np.zeros((0, 4)), np.zeros(0)
        inputs = self.sam_proc(images=image, input_boxes=[boxes.tolist()], return_tensors="pt").to(self.device)
        out = self.sam(**inputs, multimask_output=False)
        masks = self.sam_proc.post_process_masks(out.pred_masks.cpu(), inputs["original_sizes"].cpu())[0]
        masks = masks[:, 0].numpy().astype(bool)  # (N, H, W)
        return masks, boxes.astype(float), scores.astype(float)

    @torch.inference_mode()
    def segment_point(self, image, point):
        inputs = self.sam_proc(images=image, input_points=[[[list(map(float, point))]]],
                               input_labels=[[[1]]], return_tensors="pt").to(self.device)
        out = self.sam(**inputs, multimask_output=True)
        masks = self.sam_proc.post_process_masks(out.pred_masks.cpu(), inputs["original_sizes"].cpu())[0][0]
        scores = out.iou_scores.float().cpu().numpy().reshape(-1)
        order = np.argsort(-scores)[:MAX_RESULTS]
        return masks.numpy().astype(bool)[order], scores[order].astype(float)

In [ ]:
%%writefile /kaggle/working/patch_gsam2.py
"""Swap ASPIRE's SAM3 provider for Grounded-SAM 2 (Grounding DINO + SAM 2.1), keeping its HTTP API unchanged.

Usage: python patch_gsam2.py <aspire_repo> <path/to/gsam2_backend.py>
- Copies gsam2_backend.py into cap_harness/providers/sam3/.
- Rewrites service.py: the SAM3 imports, the two inference functions, and model loading. The FastAPI app,
  request/response schemas, endpoints, GPU semaphore and memory release stay byte-identical,
  so ASPIRE's client, doctor and programs see the same service.
Idempotent: refuses to patch twice, and asserts every anchor it replaces exists exactly once.
"""
import re
import shutil
import sys
from pathlib import Path

repo, backend = Path(sys.argv[1]), Path(sys.argv[2])
pkg = repo / "src/cap_harness/providers/sam3"
svc_path = pkg / "service.py"
s = svc_path.read_text()
if "GroundedSAM2" in s:
    print("already patched"); sys.exit(0)
shutil.copy(backend, pkg / "gsam2_backend.py")


def sub_once(pattern, repl, text, flags=re.S):
    new, n = re.subn(pattern, lambda m: repl, text, count=1, flags=flags)
    assert n == 1, f"anchor not found: {pattern[:60]}"
    return new


s = sub_once(r"from sam3\.model\.sam3_image_processor import Sam3Processor\nfrom sam3\.model_builder import build_sam3_image_model\n",
             "from .gsam2_backend import GroundedSAM2\n", s)

s = sub_once(r"def _segment_text\(image: Image\.Image, prompt: str\) -> SegmentResponse:\n.*?\n\n\ndef _segment_point",
             '''def _segment_text(image: Image.Image, prompt: str) -> SegmentResponse:
    masks, boxes, scores = _MODEL.segment_text(image, prompt)
    results = [
        MaskData(
            mask_base64=_encode_raw(masks[index]),
            shape=list(masks[index].shape),
            box=[float(v) for v in boxes[index]],
            score=float(scores[index]),
            label=prompt,
        )
        for index in range(len(scores))
    ]
    results.sort(key=lambda item: item.score, reverse=True)
    return SegmentResponse(results=results[:_MAX_SEGMENTATIONS])


def _segment_point''', s)

s = sub_once(r"def _segment_point\(image: Image\.Image, point: tuple\[float, float\]\) -> PointPromptResponse:\n.*?\n\n\n@app",
             '''def _segment_point(image: Image.Image, point: tuple[float, float]) -> PointPromptResponse:
    mask_array, score_array = _MODEL.segment_point(image, point)
    if mask_array.size == 0 or score_array.size == 0:
        return PointPromptResponse(
            scores=[], masks_base64="", masks_shape=[0, 0, 0], masks_dtype="float32"
        )
    mask_array = mask_array[:_MAX_SEGMENTATIONS].astype(np.float32)
    score_array = score_array[:_MAX_SEGMENTATIONS]
    return PointPromptResponse(
        scores=score_array.astype(float).tolist(),
        masks_base64=_encode_raw(mask_array),
        masks_shape=list(mask_array.shape),
        masks_dtype=str(mask_array.dtype),
    )


@app''', s)

s = sub_once(r'    LOGGER\.info\("Loading SAM3 model on %s", args\.device\)\n.*?confidence_threshold=0\.0\)\n',
             '''    LOGGER.info("Loading Grounded-SAM 2 (SAM3 stand-in) on %s", args.device)
    _MODEL = GroundedSAM2(device=args.device)
    _PROCESSOR = _MODEL
''', s)

svc_path.write_text(s)
print("patched", svc_path)

In [ ]:
# 3. NVIDIA user-space EGL libs matching the driver (MuJoCo offscreen rendering), about 2 min
ver = subprocess.check_output(["nvidia-smi", "--query-gpu=driver_version", "--format=csv,noheader"]).decode().split()[0]
run = f"NVIDIA-Linux-x86_64-{ver}.run"
for url in [f"https://us.download.nvidia.com/tesla/{ver}/{run}",
            f"https://download.nvidia.com/XFree86/Linux-x86_64/{ver}/{run}"]:
    if os.system(f"wget -q {url} -O /tmp/{run}") == 0 and os.path.getsize(f"/tmp/{run}") > 1e7:
        print("downloaded", url); break
else:
    raise SystemExit(f"driver {ver} .run not found; tell Claude")
os.system(f"sh /tmp/{run} --silent --no-kernel-module --no-kernel-modules --no-x-check --no-nouveau-check "
          f"--no-systemd --no-nvidia-modprobe --no-drm --no-questions --ui=none --install-libglvnd > /tmp/nv.log 2>&1")
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y libegl1 libgl1 ffmpeg > /dev/null 2>&1
!ls /usr/lib/x86_64-linux-gnu/ | grep -E "libEGL_nvidia|libEGL.so.1" | head -3

In [ ]:
# 4. ASPIRE at the commit we read, plus 2 small patches for the T4:
#    (a) a 't4' hardware profile (ASPIRE only ships 4090/5090/L40/5880 profiles)
#    (b) SAM3 autocast bf16 -> fp16 (T4 has no bf16). Set SAM3_DTYPE=float32 in cell 1 to disable autocast.
!pip -q install -U uv
if not os.path.exists(REPO):
    !git clone -q https://github.com/NVlabs/ASPIRE.git {REPO}
!cd {REPO} && git checkout -q {COMMIT} && git log -1 --format='ASPIRE %h %cd'
prof = {"schema": "cap-harness/profile/1", "name": "t4", "arch": "sm_75", "venv_root": None,
        "description": "Kaggle Tesla T4 (sm_75). SAM3 on the last GPU, MuJoCo EGL rendering on GPU 0.",
        "gpus": {"sam3": NGPU - 1, "contact_graspnet": NGPU - 1, "curobo": 0},
        "concurrency": {"sam3": 1, "contact_graspnet": 1, "pyroki": 4, "curobo": 1}, "ports": {}}
json.dump(prof, open(f"{REPO}/configs/profiles/t4.json", "w"), indent=2)
lock_p = f"{REPO}/configs/dependency-lock.json"; lock = json.load(open(lock_p))
lock["gpu_runtime"]["arches"]["sm_75"] = {"cuda_arch_list": "7.5", "profiles": ["t4"]}
json.dump(lock, open(lock_p, "w"), indent=2)
svc_p = f"{REPO}/src/cap_harness/providers/sam3/service.py"; svc = open(svc_p).read()
if PERCEPTION == "gsam2":
    !python3 /kaggle/working/patch_gsam2.py {REPO} /kaggle/working/gsam2_backend.py
elif "_SAM3_DTYPE" not in svc:
    svc = svc.replace("import torch\n", "import torch\nimport os as _os\n"
                      "_SAM3_DTYPE = getattr(torch, _os.environ.get('SAM3_DTYPE', 'float16'))\n"
                      "_SAM3_AUTOCAST = _os.environ.get('SAM3_DTYPE', 'float16') != 'float32'\n", 1)
    svc = svc.replace("torch.autocast(device_type, dtype=torch.bfloat16)",
                      "torch.autocast(device_type, dtype=_SAM3_DTYPE, enabled=_SAM3_AUTOCAST)")
    open(svc_p, "w").write(svc)
# (c) ASPIRE bug: `elif not init_states:` crashes when init_states is a numpy array (saved-scene mode)
ad_p = f"{REPO}/src/cap_harness/libero/adapter.py"; ad = open(ad_p).read()
ad = ad.replace("        elif not init_states:", "        elif len(init_states) == 0:"); open(ad_p, "w").write(ad)
print("adapter init_states fix:", "len(init_states) == 0" in ad)
# (d) settle window for the final waypoint of every move (see SETTLE_STEPS in cell 1)
ad = open(ad_p).read()
ad = ad.replace("TRAJECTORY_MAX_STEPS_PER_WAYPOINT = 120", f"TRAJECTORY_MAX_STEPS_PER_WAYPOINT = {SETTLE_STEPS}")
open(ad_p, "w").write(ad)
print("settle steps:", [l for l in ad.splitlines() if l.startswith("TRAJECTORY_MAX_STEPS_PER_WAYPOINT")])
assert f"TRAJECTORY_MAX_STEPS_PER_WAYPOINT = {SETTLE_STEPS}" in ad, "settle patch did not apply"
print("perception:", PERCEPTION, "| service uses Grounded-SAM 2:", "GroundedSAM2" in open(svc_p).read(),
      "| sam3 fp16 autocast sites:", open(svc_p).read().count("_SAM3_AUTOCAST)"))

In [ ]:
# 5. Install (pinned by ASPIRE's lock): LIBERO client env + SAM3 + pyroki providers, about 15-20 min
!uv python install 3.11 -q
PY311 = subprocess.check_output(["uv", "python", "find", "3.11"]).decode().strip()
os.environ["CAP_HARNESS_PYTHON"] = PY311; print("python", PY311)
t0 = time.time()
!cd {REPO} && scripts/bootstrap_libero.sh > /tmp/boot_libero.log 2>&1; echo "libero exit $?"; tail -3 /tmp/boot_libero.log
!cd {REPO} && scripts/bootstrap_providers.sh --providers sam3,pyroki --profile t4 > /tmp/boot_prov.log 2>&1; echo "providers exit $?"; tail -3 /tmp/boot_prov.log
if PERCEPTION == "gsam2":  # the sam3 venv keeps ASPIRE's pinned torch; add transformers for Grounding DINO + SAM 2.1
    !uv pip install -q --python {REPO}/.venv-sam3/bin/python "transformers==4.57.6"
    !{REPO}/.venv-sam3/bin/python -c "from transformers import Sam2Model, AutoModelForZeroShotObjectDetection; print('gsam2 imports OK')"
print(f"install took {(time.time() - t0) / 60:.1f} min")
!{REPO}/.venv-libero/bin/python -c "import torch, mujoco; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_capability(0), 'kernel test', torch.ones(4, device='cuda').sum().item(), '| mujoco', mujoco.__version__)"

In [ ]:
# 6. Start the provider services in the background (SAM3 downloads ~3.5 GB of gated weights on first start)
# In gsam2 mode the 'sam3' provider slot serves Grounded-SAM 2 (first start downloads ~1.2 GB of open weights)
PROVIDERS = "sam3,pyroki"
!pkill -f supervise_services.sh; sleep 1
subprocess.Popen(f"cd {REPO} && nohup scripts/supervise_services.sh --profile t4 --providers {PROVIDERS} "
                 f"--ready-timeout 1200 > /root/services.log 2>&1 &", shell=True)
for i in range(130):
    log = open("/root/services.log").read() if os.path.exists("/root/services.log") else ""
    if "all provider services ready" in log or "error:" in log:
        break
    time.sleep(10)
print(log[-1500:])
if "error:" in log:
    !tail -40 {REPO}/validation-artifacts/service-logs/sam3.log
    !tail -20 {REPO}/validation-artifacts/service-logs/pyroki.log
    raise SystemExit("services failed; send Claude the output above")

In [ ]:
# 7. ASPIRE's own health checks: providers unit test + one LIBERO reset
os.chdir(REPO)
CAP = f"{REPO}/.venv-libero/bin/cap-harness"
os.environ["VIRTUAL_ENV"] = f"{REPO}/.venv-libero"
r1 = os.system(f"{CAP} doctor --providers {PROVIDERS} --unit --output {OUT}/doctor_unit.json")
r2 = os.system(f"{CAP} doctor --reset --providers {PROVIDERS} --output {OUT}/doctor_reset.json")
!nvidia-smi --query-gpu=index,memory.used --format=csv
assert r1 == 0 and r2 == 0, f"doctor FAILED (unit={r1}, reset={r2}); stopping before any Claude call"
print("doctor: OK")

In [ ]:
# 8. Claude Code CLI (uses your plan via CLAUDE_CODE_OAUTH_TOKEN) + a 1-token check
!curl -fsSL https://claude.ai/install.sh | bash > /tmp/claude_install.log 2>&1; tail -1 /tmp/claude_install.log
!claude --version
for attempt in range(3):
    chk = subprocess.run('echo "Reply with exactly: CLAUDE OK" | claude -p --model sonnet --tools "" --strict-mcp-config 2>&1',
                         shell=True, capture_output=True, text=True).stdout
    print(chk[-300:])
    if "CLAUDE OK" in chk:
        break
    time.sleep(30)
else:
    raise SystemExit("Claude check failed (token or usage limit); stopping before the study")

In [ ]:
%%writefile /kaggle/working/agent_pilot.py
"""SkillMem pilot: does a robot's skill memory help, hurt, or confuse an LLM that writes robot programs?

For each task and each memory condition, Claude writes ONE program (single call, no tools). The program
runs on 3 development seeds; Claude gets ONE repair call with the failures; both versions are scored on
5 held-out seeds. Everything runs through ASPIRE's harness (cap-harness) on LIBERO-Pro.

Memory conditions (the only thing that changes between conditions):
  A  none      API reference only
  B  relevant  + ASPIRE's tabletop skill library (grasp / localize / transport / manipulation)
  C  bloated   + B + ASPIRE's BEHAVIOR (mobile humanoid) skills + its evosearch pipeline guide
  D  managed   the same pool as C, but a BM25 retriever keeps only the sections most relevant to the
               task, up to the size of B

Usage: python agent_pilot.py --tasks libero_goal_swap:8 --conds A,B,C,D --out /kaggle/working/run1
Resumable: every finished step is recorded in <out>/state.json and skipped on rerun.
"""
import argparse
import glob
import json
import math
import os
import pathlib
import re
import subprocess
import sys
import time
from collections import Counter

REPO = os.environ.get("ASPIRE_REPO", "/root/aspire")
CAP = f"{REPO}/.venv-libero/bin/cap-harness"
SK = f"{REPO}/.claude/skills"
RELEVANT = [f"{SK}/iterative-debugging/skills/{n}.md" for n in ("grasp", "localize", "transport", "manipulation")]
DISTRACT = sorted(glob.glob(f"{SK}/iterative-debugging/behavior/skills/*.md")) + [f"{SK}/evosearch/skills/evosearch-iteration.md"]
DEV_SEEDS, HELDOUT_SEEDS = [51, 52, 53], [6, 7, 8, 9, 10]
MAX_STEPS = int(os.environ.get("SKILLMEM_MAX_STEPS", "2500"))  # Run 3: 1000 -> 2500 (Run 2 episodes ran out of ticks)

EP_TIMEOUT = int(os.environ.get("SKILLMEM_EP_TIMEOUT", "700"))  # slowest Run 3 episode: 270 s
SYSTEM = """You write robot control programs for ASPIRE's code-as-policy harness: a Franka Panda arm in LIBERO-Pro (MuJoCo).
Rules for every program (violations make the run invalid):
- Use only the documented generated-program API. `import math` is the only import allowed; no NumPy, files, network, or simulator internals.
- Perceive the scene at runtime (e.g. localize objects with the SAM3 tools). Never hard-code object coordinates: every seed has a different layout.
- The episode stops at 1000 control ticks (20 Hz). Budget your motions.
- Set a top-level `result` dict recording what each stage achieved (e.g. {"localized": True, "grasped": False, ...}).
Reply with exactly one ```python code block containing the complete program and nothing else."""


# ---------------------------------------------------------------- memory
def strip_frontmatter(text):
    return re.sub(r"\A---\n.*?\n---\n", "", text, flags=re.S)


def sections(paths):
    """Split markdown files into '## ' sections: (source, title, text)."""
    out = []
    for p in paths:
        text = strip_frontmatter(open(p).read())
        parts = re.split(r"(?m)^(?=## )", text)
        for part in parts:
            if part.strip():
                title = part.strip().splitlines()[0][:80]
                out.append((os.path.relpath(p, SK), title, part.strip()))
    return out


def tokenize(s):
    return re.findall(r"[a-z]+", s.lower())


STOP = set("the a an of to in on and or for with is are be it this that from by as at into onto your you".split())


def task_query(language, bddl):
    """Retrieval query: the task sentence + object/fixture names from the BDDL (not its boilerplate)."""
    names = []
    for block in re.findall(r"\(:(?:objects|fixtures)(.*?)\)", bddl, flags=re.S):
        names += re.findall(r"([a-z_]+?)_\d+\s*-", block)
    return language + " " + " ".join(n.replace("_", " ") for n in names)


def bm25_rank(query, docs, k1=1.5, b=0.75):
    toks = [[w for w in tokenize(d) if w not in STOP] for d in docs]
    avg = sum(map(len, toks)) / max(1, len(toks))
    df = Counter(w for t in toks for w in set(t))
    n = len(docs)
    q = set(tokenize(query)) - STOP
    scores = []
    for t in toks:
        tf = Counter(t)
        s = 0.0
        for w in q:
            if w in tf:
                idf = math.log(1 + (n - df[w] + 0.5) / (df[w] + 0.5))
                s += idf * tf[w] * (k1 + 1) / (tf[w] + k1 * (1 - b + b * len(t) / avg))
        scores.append(s)
    return sorted(range(n), key=lambda i: -scores[i]), scores


def render(secs):
    return "\n\n".join(f"<!-- {src} -->\n{text}" for src, _, text in secs)


def build_memory(cond, query):
    rel, dis = sections(RELEVANT), sections(DISTRACT)
    if cond == "A":
        return "", []
    if cond == "B":
        return render(rel), [f"{s}|{t}" for s, t, _ in rel]
    pool = rel + dis
    if cond == "C":
        pool = sorted(pool, key=lambda x: x[0])  # deterministic, relevant and distractor files interleaved by path
        return render(pool), [f"{s}|{t}" for s, t, _ in pool]
    if cond == "D":
        budget = len(render(rel))
        # titles count three times: they say what a section is for
        order, scores = bm25_rank(query, [f"{t} {t} {t} {x}" for _, t, x in pool])
        picked, size = [], 0
        for i in order:
            if scores[i] <= 0:
                break  # keep only sections the task actually matches
            if picked and size + len(pool[i][2]) > budget:
                continue
            picked.append(pool[i]); size += len(pool[i][2])
        return render(picked), [f"{s}|{t}|{scores[pool.index(x)]:.1f}" for x in picked for s, t, _ in [x]]
    raise ValueError(cond)


# ---------------------------------------------------------------- task info
def task_info(task_ref):
    code = ("import json,sys\nfrom cap_harness.validation import load_libero_registry\n"
            f"m = load_libero_registry().resolve({task_ref!r})\n"
            "print(json.dumps({'language': m.language, 'task_name': m.task_name, 'bddl': open(m.bddl_path).read()}))")
    out = subprocess.run([f"{REPO}/.venv-libero/bin/python", "-c", code], capture_output=True, text=True, cwd=REPO)
    line = [l for l in out.stdout.splitlines() if l.startswith("{")]
    if not line:
        raise RuntimeError(f"task_info failed for {task_ref}: {out.stderr[-800:]}")
    return json.loads(line[-1])


# ---------------------------------------------------------------- claude
def claude(prompt, model):
    cmd = ["claude", "-p", "--model", model, "--tools", "", "--system-prompt", SYSTEM,
           "--output-format", "json", "--no-session-persistence"]
    t0 = time.time()
    r = subprocess.run(cmd, input=prompt, capture_output=True, text=True, timeout=1200)
    try:
        j = json.loads(r.stdout)
    except json.JSONDecodeError:
        raise RuntimeError(f"claude returned no JSON (rc={r.returncode}): {r.stdout[-500:]} {r.stderr[-500:]}")
    if j.get("is_error"):
        raise RuntimeError(f"claude error: {str(j.get('result'))[:500]}")
    text = j.get("result", "")
    blocks = re.findall(r"```python\n(.*?)```", text, flags=re.S)
    u = j.get("usage", {})
    usage = {"input": u.get("input_tokens", 0), "cache_read": u.get("cache_read_input_tokens", 0),
             "cache_write": u.get("cache_creation_input_tokens", 0), "output": u.get("output_tokens", 0),
             "cost_usd_equiv": j.get("total_cost_usd"), "seconds": round(time.time() - t0, 1)}
    return (blocks[-1] if blocks else ""), usage


# ---------------------------------------------------------------- run + feedback
def run_program(prog_path, suite, tid, seed, out_root, videos):
    out_root = pathlib.Path(out_root)
    cmd = [CAP, "run", "--benchmark", "libero-pro", "--suite", suite, "--task-id", str(tid), "--seed", str(seed),
           "--init-mode", "seeded", "--max-steps", str(MAX_STEPS), "--program", str(prog_path),
           "--output-root", str(out_root), "--flat-run-dir"] + ([] if videos else ["--no-videos"])
    log = out_root.with_suffix(".log")
    out_root.parent.mkdir(parents=True, exist_ok=True)
    with open(log, "w") as f:
        try:
            rc = subprocess.run(cmd, stdout=f, stderr=subprocess.STDOUT, cwd=REPO, timeout=EP_TIMEOUT).returncode
        except subprocess.TimeoutExpired:
            return {"seed": seed, "success": False, "error": f"program killed after {EP_TIMEOUT}s (endless loop?)", "rc": -9}
    outs = sorted(out_root.rglob("outcome.json"))
    if not outs:
        return {"seed": seed, "success": False, "error": "no outcome", "log_tail": log.read_text(errors='replace')[-600:], "rc": rc}
    pr_path = outs[-1].parent / "source/program-result.json"
    try:
        o = json.load(open(outs[-1]))
        pr = json.load(open(pr_path)) if pr_path.exists() else {}
    except ValueError as e:
        return {"seed": seed, "success": False, "error": f"unreadable outcome: {e}", "rc": rc}
    # Run 3: program-result.json has no error text, so read it from the per-call traces (trace/calls/*/output.json)
    msgs = []
    for f in sorted((outs[-1].parent / "trace/calls").glob("*/output.json")):
        try:
            d = json.load(open(f))
        except Exception:
            continue
        e = d.get("error") if isinstance(d, dict) else None
        if isinstance(e, dict) and e.get("message"):
            det = e.get("details") or {}
            extra = f" (joint_error_rad={det['joint_error_rad']:.3f}, steps_used={d.get('steps_executed')})" if isinstance(det.get("joint_error_rad"), (int, float)) else ""
            msgs.append(e["message"][:140] + extra)
    failed = [{"tool": c.get("tool"), "code": c.get("failure_code") or c.get("error_type"),
               "msg": (c.get("error_message") or "")[:160]} for c in pr.get("calls", []) if not c.get("ok")]
    seen = []  # each failure is traced at two layers: drop exact consecutive repeats
    for m in msgs:
        if not seen or seen[-1] != m:
            seen.append(m)
    return {"seed": seed, "success": bool(o.get("task_success")), "termination": o.get("termination_reason"),
            "steps": o.get("steps_executed"), "wall_s": round(o.get("wall_duration_s") or 0),
            "program_error": (str(pr.get("error"))[:600] if pr.get("error") else None),
            "result": json.dumps(pr.get("result"))[:700], "failed_calls": failed[:8], "errors": seen if len(seen) <= 8 else seen[:4] + seen[-4:], "rc": rc}


def feedback(runs):
    lines = []
    for r in runs:
        lines.append(f"seed {r['seed']}: success={r['success']} termination={r.get('termination')} steps={r.get('steps')} of {MAX_STEPS}")
        if r.get("program_error"):
            lines.append(f"  program error: {r['program_error']}")
        if r.get("error"):
            lines.append(f"  harness error: {r['error']} {r.get('log_tail', '')}")
        lines.append(f"  result: {r.get('result')}")
        for c in r.get("failed_calls", []):
            lines.append(f"  failed call: {c['tool']} -> {c['code']} {c['msg']}")
        for m in r.get("errors", []):
            lines.append(f"  error: {m}")
    return "\n".join(lines)


# ---------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--tasks", required=True)  # comma list of suite:id
    ap.add_argument("--conds", default="A,B,C,D")
    ap.add_argument("--out", required=True)
    ap.add_argument("--model", default="sonnet")
    ap.add_argument("--max-calls", type=int, default=40)  # hard cap on Claude calls this session
    args = ap.parse_args()
    out = pathlib.Path(args.out); out.mkdir(parents=True, exist_ok=True)
    sp = out / "state.json"
    state = json.load(open(sp)) if sp.exists() else {}
    save = lambda: json.dump(state, open(sp, "w"), indent=1)
    api = open(f"{REPO}/docs/api-reference.md").read()
    calls = 0

    for task_ref in args.tasks.split(","):
        suite, tid = task_ref.split(":")
        info = state.setdefault(f"info:{task_ref}", task_info(task_ref)); save()
        query = task_query(info["language"], info["bddl"])
        for cond in args.conds.split(","):
            key = f"{task_ref}|{cond}"
            st = state.setdefault(key, {}); tag = f"{suite}_{tid}_{cond}"
            mem, mem_index = build_memory(cond, query)
            st["memory_chars"], st["memory_sections"] = len(mem), mem_index
            context = (f"# API reference\n{api}\n\n" + (f"# Skill library (lessons from earlier robot tasks)\n{mem}\n\n" if mem else "")
                       + f"# Task\nsuite={suite} task_id={tid}\nlanguage: {info['language']}\nBDDL scene definition:\n{info['bddl']}\n")
            for ver in ("v1", "v2"):
                if ver == "v2" and st.get("v1_dev_success") == len(DEV_SEEDS):
                    st["v2_skipped"] = "v1 passed every dev seed"; save(); break
                if f"{ver}_program" not in st:
                    if calls >= args.max_calls:
                        print(f"[stop] reached --max-calls {args.max_calls}; rerun to resume"); save(); return
                    if ver == "v1":
                        prompt = context + "\nWrite the program for this task."
                    else:
                        prompt = (context + f"\n# Your previous program\n```python\n{st['v1_program']}\n```\n"
                                  f"\n# What happened on the development seeds\n{feedback(st['v1_dev'])}\n"
                                  "\nFix the program. Return the complete corrected program.")
                    try:
                        prog, usage = claude(prompt, args.model)
                    except Exception as e:
                        print(f"[stop] {key} {ver}: {e}"); save(); return  # e.g. usage limit: resume later
                    calls += 1
                    st[f"{ver}_program"], st[f"{ver}_usage"], st[f"{ver}_prompt_chars"] = prog, usage, len(prompt)
                    save(); print(f"[gen] {key} {ver}: {len(prog)} chars, usage {usage}")
                prog_path = out / "programs" / f"{tag}_{ver}.py"; prog_path.parent.mkdir(exist_ok=True)
                prog_path.write_text(st[f"{ver}_program"] or "result = {'empty_program': True}\n")
                for phase, seeds, videos in (("dev", DEV_SEEDS, False), ("heldout", HELDOUT_SEEDS, True)):
                    if ver == "v2" and phase == "dev":
                        continue
                    k = f"{ver}_{phase}"
                    st.setdefault(k, [])
                    done = {r["seed"] for r in st[k]}
                    for s in seeds:
                        if s in done:
                            continue
                        r = run_program(prog_path, suite, tid, s, out / "runs" / tag / ver / f"{phase}_s{s}", videos)
                        st[k].append(r); save()
                        print(f"[run] {key} {ver} {phase} seed {s}: success={r['success']} ({r.get('termination')}, {r.get('wall_s')}s)")
                    st[f"{k}_success"] = sum(r["success"] for r in st[k]); save()
            print(f"[done] {key}: v1 dev {st.get('v1_dev_success')}/{len(DEV_SEEDS)}, v1 heldout "
                  f"{st.get('v1_heldout_success')}/{len(HELDOUT_SEEDS)}, v2 heldout {st.get('v2_heldout_success', '-')}")
    print("PILOT COMPLETE")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/agent_study.py
"""SkillMem Run 3 (Run 2 + 2500-tick budget + real error messages): does a robot's skill memory help, hurt, or confuse an LLM that writes robot programs?

Changes from the pilot (agent_pilot.py, whose memory/run helpers this file reuses):
  1. The sandbox rules (ASPIRE's program validator) are in the system prompt for EVERY condition. In the
     pilot only condition D happened to retrieve them, which confounded the comparison.
  2. Up to 3 repair rounds (v2..v4) instead of 1. Each version runs on the 3 dev seeds; we stop as soon
     as a version passes all 3. v1 (one-shot) and the final version are both scored on 5 held-out seeds.
  3. Each repair also shows Claude the last camera frame (agentview) of the first failed dev seed.
  4. MCP connectors are disabled (--strict-mcp-config): fewer tokens, and the agent can't see other tools.

Usage: python agent_study.py --tasks libero_goal_swap:8,libero_goal_swap:1 --conds A,B,C,D --out DIR
Resumable through DIR/state.json.
"""
import argparse
import base64
import io
import json
import os
import pathlib
import re
import shutil
import subprocess
import time

from agent_pilot import (DEV_SEEDS, HELDOUT_SEEDS, REPO, build_memory, feedback, run_program, task_info,
                         task_query)

DEADLINE = float(os.environ.get("SKILLMEM_DEADLINE", "inf"))  # unix time: stop cleanly before the GPU quota ends
RETRY_WAITS = [30, 120] + [600] * 8  # transient errors and 5-hour-limit resets
MAX_REPAIRS = 5  # Run 3: 3 -> 5 repair rounds (v2..v6)

SYSTEM = """You write robot control programs for ASPIRE's code-as-policy harness: a Franka Panda arm in LIBERO-Pro (MuJoCo).
The program is checked by a sandbox validator before it runs. Violations reject the whole program:
- No name or attribute may start with an underscore (no `_helper`, no `obj.__dict__`).
- Not allowed: class, with, global, nonlocal, del, async/await, `from x import y`. The only import allowed is `import math` (no alias).
- Only these builtins exist: Exception ValueError TypeError abs all any bool dict enumerate float int len list max min print range reversed round set sorted str sum tuple zip. (No getattr, hasattr, isinstance, type, open, map, filter.)
- Contract constructors available by name: ArmCommand MotionStrategy Pose RobotAction SynchronizedTrajectory Trajectory. Everything else comes from the documented API; use only fields and functions the API reference documents.
Task rules:
- Perceive the scene at runtime (localize objects with the segmentation tools). Never hard-code object coordinates: object positions differ on every seed.
- The episode stops at 2500 control ticks (20 Hz). Every control tick spent waiting counts, including the ticks a failed move burns. After the episode ends, every further call fails with "executing action in terminated episode".
- The arm moves about 0.15 rad/s per joint, so large rotations cost many ticks. Keep the gripper pointing down and avoid big wrist turns unless the task needs them.
- A move_to_pose "timeout" with a large joint_error_rad (above about 0.05) means the arm is BLOCKED, not slow: the target is inside the table or an object, or out of reach. Retrying the same target wastes ticks. Raise it, approach from above, or change it.
- Moves are straight joint interpolation with NO obstacle avoidance. cuRobo is not installed, so never request it.
  The scene has tall obstacles (wine rack, cabinet, bottle). Never move sideways low: first go straight up to
  z of about 0.30 m (0.28-0.35, robot_base), travel at that height in short hops, then descend vertically over the target.
  The hand housing is much wider than the fingers: if a descent is blocked by a neighbour, choose another contact point or yaw the gripper 90 degrees.
  If a hop is blocked, back up, then go higher or around the obstacle.
- `math` is not predefined: write `import math` if you use it.
- IMPORTANT: move_to_pose fails with status "timeout" whenever the final joint error is above 0.01 rad, whatever `tolerance` you pass.
  An error below 0.05 rad means the arm has practically ARRIVED: continue the plan, don't retry or abort. Test it like this:
  `near = r.ok or (r.error is not None and r.error.details.get("joint_error_rad", 9) < 0.05)`
- Sanity-check every localization: small objects are sometimes confused with the robot's own gripper or with other objects.
  A stove knob or a plate is near the table (z < 0.15) and a knob is next to the stove. Try other queries or cameras if not.
- Check every motion result (with the near test above) and adapt on a real failure. Don't run the rest of the plan blind.
- Set a top-level `result` dict recording what each stage achieved (e.g. {"localized": True, "grasped": False, ...}).
Reply with exactly one ```python code block containing the complete program and nothing else."""


def png_b64(path, max_w=640):
    from PIL import Image
    img = Image.open(path).convert("RGB")
    if img.width > max_w:
        img = img.resize((max_w, round(img.height * max_w / img.width)))
    buf = io.BytesIO(); img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()


def claude(prompt, model, image=None):
    content = []
    if image:
        content.append({"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": png_b64(image)}})
    content.append({"type": "text", "text": prompt})
    msg = {"type": "user", "message": {"role": "user", "content": content}}
    cmd = ["claude", "-p", "--model", model, "--tools", "", "--strict-mcp-config", "--system-prompt", SYSTEM,
           "--input-format", "stream-json", "--output-format", "stream-json", "--verbose", "--no-session-persistence"]
    t0 = time.time()
    r = subprocess.run(cmd, input=json.dumps(msg) + "\n", capture_output=True, text=True, timeout=600)
    finals = [json.loads(l) for l in r.stdout.splitlines() if l.startswith("{") and '"type":"result"' in l.replace(" ", "")]
    if not finals:
        raise RuntimeError(f"claude gave no result (rc={r.returncode}): {r.stdout[-400:]} {r.stderr[-400:]}")
    j = finals[-1]
    if j.get("is_error"):
        raise RuntimeError(f"claude error: {str(j.get('result'))[:400]}")
    blocks = re.findall(r"```python\n(.*?)```", j.get("result", ""), flags=re.S)
    u = j.get("usage", {})
    usage = {"input": u.get("input_tokens", 0), "cache_read": u.get("cache_read_input_tokens", 0),
             "cache_write": u.get("cache_creation_input_tokens", 0), "output": u.get("output_tokens", 0),
             "cost_usd_equiv": j.get("total_cost_usd"), "seconds": round(time.time() - t0, 1), "image": bool(image)}
    prog = blocks[-1] if blocks else ""
    if re.search(r"\bmath\.", prog) and not re.search(r"^import math\b", prog, flags=re.M):
        prog = "import math\n" + prog
    return prog, usage


def claude_retry(prompt, model, image):
    for wait in RETRY_WAITS + [None]:
        try:
            prog, usage = claude(prompt, model, image)
            if prog.strip():
                return prog, usage
            err = RuntimeError("reply had no python block")
        except Exception as e:
            err = e
        if wait is None or time.time() + wait > DEADLINE:
            raise err
        print(f"[retry] claude: {str(err)[:200]}; waiting {wait}s", flush=True)
        time.sleep(wait)


def last_frame(run_root):
    """agentview.png of the last keyframe ASPIRE recorded for this run (the scene after the final motion)."""
    frames = sorted(pathlib.Path(run_root).rglob("media/keyframes/*/agentview.png"))
    return str(frames[-1]) if frames else None


def keep_frame(src, out, name):
    """Copy the frame into out/frames/ (checkpoints skip the big keyframe folders)."""
    if not src or not pathlib.Path(src).exists():
        return None
    dst = out / "frames" / f"{name}.png"; dst.parent.mkdir(exist_ok=True)
    shutil.copy(src, dst)
    return str(dst)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--tasks", required=True)
    ap.add_argument("--conds", default="A,B,C,D")
    ap.add_argument("--out", required=True)
    ap.add_argument("--model", default="sonnet")
    ap.add_argument("--max-calls", type=int, default=120)
    args = ap.parse_args()
    out = pathlib.Path(args.out); out.mkdir(parents=True, exist_ok=True)
    sp = out / "state.json"
    state = json.load(open(sp)) if sp.exists() else {}
    def save():  # atomic, so a checkpoint never packs a half-written state
        tmp = out / "state.json.tmp"
        tmp.write_text(json.dumps(state, indent=1))
        os.replace(tmp, sp)
    api = open(f"{REPO}/docs/api-reference.md").read()
    for v in state.values():  # move frames recorded before this fix into out/frames/
        for k, rs in (v.items() if isinstance(v, dict) else []):
            if isinstance(rs, list):
                for r in rs:
                    if isinstance(r, dict) and r.get("frame") and "/frames/" not in r["frame"]:
                        r["frame"] = keep_frame(r["frame"], out, f"old_{abs(hash(r['frame']))}")
    save()
    calls = 0
    versions = [f"v{i}" for i in range(1, MAX_REPAIRS + 2)]

    def evaluate(st, ver, phase, seeds, tag, suite, tid, videos):
        k = f"{ver}_{phase}"
        st.setdefault(k, [])
        done = {r["seed"] for r in st[k]}
        prog_path = out / "programs" / f"{tag}_{ver}.py"; prog_path.parent.mkdir(exist_ok=True)
        prog_path.write_text(st[f"{ver}_program"] or "result = {'empty_program': True}\n")
        for s in seeds:
            if s in done:
                continue
            if time.time() > DEADLINE:
                save(); print("[stop] time budget used up; rerun to resume", flush=True); raise SystemExit(0)
            root = out / "runs" / tag / ver / f"{phase}_s{s}"
            r = run_program(prog_path, suite, tid, s, root, videos)
            r["frame"] = keep_frame(last_frame(root), out, f"{tag}_{ver}_{phase}_s{s}")
            for kind in ("keyframes", "overlays"):  # gigabytes; the last frame was copied above
                for d in list(root.rglob(f"media/{kind}")):
                    shutil.rmtree(d, ignore_errors=True)
            st[k].append(r); save()
            print(f"[run] {tag} {ver} {phase} seed {s}: success={r['success']} ({r.get('termination')}, {r.get('wall_s')}s)", flush=True)
        st[f"{k}_success"] = sum(r["success"] for r in st[k]); save()

    for task_ref in args.tasks.split(","):
        suite, tid = task_ref.split(":")
        if f"info:{task_ref}" not in state:
            state[f"info:{task_ref}"] = task_info(task_ref); save()
        info = state[f"info:{task_ref}"]
        query = task_query(info["language"], info["bddl"])
        for cond in args.conds.split(","):
            key, tag = f"{task_ref}|{cond}", f"{suite}_{tid}_{cond}"
            st = state.setdefault(key, {})
            if st.get("finished"):
                continue
            mem, mem_index = build_memory(cond, query)
            st["memory_chars"], st["memory_sections"] = len(mem), mem_index
            context = (f"# API reference\n{api}\n\n" + (f"# Skill library (lessons from earlier robot tasks)\n{mem}\n\n" if mem else "")
                       + f"# Task\nsuite={suite} task_id={tid}\nlanguage: {info['language']}\nBDDL scene definition:\n{info['bddl']}\n")
            prev = None
            for ver in versions:
                if f"{ver}_program" not in st:
                    if calls >= args.max_calls:
                        print(f"[stop] reached --max-calls {args.max_calls}; rerun to resume", flush=True); save(); return
                    image = None
                    if prev is None:
                        prompt = context + "\nWrite the program for this task."
                    else:
                        failed = [r for r in st[f"{prev}_dev"] if not r["success"]]
                        image = failed[0].get("frame") if failed else None
                        if image and not pathlib.Path(image).exists():
                            image = None
                        prompt = (context + f"\n# Your previous program ({prev})\n```python\n{st[f'{prev}_program']}\n```\n"
                                  f"\n# What happened on the development seeds\n{feedback(st[f'{prev}_dev'])}\n"
                                  + ("\nThe attached image is the scene camera (agentview) after the last motion of the first failed seed.\n" if image else "")
                                  + "\nFix the program. Return the complete corrected program.")
                    try:
                        prog, usage = claude_retry(prompt, args.model, image)
                    except Exception as e:
                        print(f"[stop] {key} {ver}: {e}", flush=True); save(); return  # e.g. usage limit: rerun later
                    calls += 1
                    st[f"{ver}_program"], st[f"{ver}_usage"] = prog, usage; save()
                    print(f"[gen] {key} {ver}: {len(prog)} chars, usage {usage}", flush=True)
                evaluate(st, ver, "dev", DEV_SEEDS, tag, suite, tid, videos=False)
                prev = ver
                if st[f"{ver}_dev_success"] == len(DEV_SEEDS):
                    break
            tried = [v for v in versions if f"{v}_dev_success" in st]
            prev = max(tried, key=lambda v: (st[f"{v}_dev_success"], int(v[1:])))
            st["final_version"] = prev
            evaluate(st, "v1", "heldout", HELDOUT_SEEDS, tag, suite, tid, videos=True)
            if prev != "v1":
                evaluate(st, prev, "heldout", HELDOUT_SEEDS, tag, suite, tid, videos=True)
            st["finished"] = True; save()
            print(f"[done] {key}: dev passes by version {[st.get(v + '_dev_success') for v in versions if v + '_dev_success' in st]}, "
                  f"v1 heldout {st['v1_heldout_success']}/5, final {prev} heldout {st[prev + '_heldout_success']}/5", flush=True)
    print("STUDY COMPLETE", flush=True)


if __name__ == "__main__":
    main()


In [ ]:
# 9. Pick the 3 weak libero_goal_swap tasks by their language
lister = (
    "from cap_harness.validation import load_libero_registry\n"
    "r = load_libero_registry()\n"
    "for i in range(10):\n"
    "    m = r.resolve(f'libero_goal_swap:{i}'); print(i, m.language)\n")
out = subprocess.run([f"{REPO}/.venv-libero/bin/python", "-c", lister], capture_output=True, text=True, cwd=REPO).stdout
print(out)
lang = {int(l.split()[0]): l.split(" ", 1)[1].lower() for l in out.splitlines() if l[:1].isdigit()}
WANT = ["bowl on the stove", "push the plate", "turn on the stove"]
TASKS = []
for w in WANT:
    hit = [i for i, s in lang.items() if w in s]
    if hit:
        TASKS.append(f"libero_goal_swap:{hit[0]}")
print("TASKS =", TASKS)
assert len(TASKS) == 3, "task languages changed; tell Claude"

In [ ]:
# 10. The study. Resumable: finished Claude calls and episodes are skipped on rerun.
os.environ["ASPIRE_REPO"] = REPO
STUDY = f"{OUT}/study"
import threading
watch_stop = threading.Event()
def watchdog():
    while not watch_stop.wait(20):
        if subprocess.run(["pgrep", "-f", "supervise_services.sh"], capture_output=True).returncode != 0:
            print("[watchdog] provider services died; restarting", flush=True)
            subprocess.Popen(["bash", "-c", f"cd {REPO} && exec scripts/supervise_services.sh --profile t4 --providers {PROVIDERS} "
                              "--ready-timeout 1200 >> /root/services.log 2>&1"])
threading.Thread(target=watchdog, daemon=True).start()
t0 = time.time()
p = subprocess.Popen(["python3", "-u", "/kaggle/working/agent_study.py", "--tasks", ",".join(TASKS), "--conds", "A,C",
                      "--out", STUDY, "--model", "sonnet", "--max-calls", "40"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd="/kaggle/working")
last_save = time.time()
for line in p.stdout:
    print(line, end="")
    try:  # a failed upload must never kill the study
        if line.startswith("[gen]"):
            ckpt.save(RUN, CK_PATHS, "gen", small=True)          # a few KB: never lose a Claude call
        elif line.startswith("[done]"):
            ckpt.pack(RUN, CK_PATHS)
            ckpt.save(RUN, CK_PATHS, "done"); last_save = time.time()
    except Exception as e:
        print("[ckpt] save failed (continuing):", e)
p.wait()
ckpt.pack(RUN, CK_PATHS)
ckpt.save(RUN, CK_PATHS, "study-end")
print(f"study session took {(time.time() - t0) / 60:.1f} min, exit {p.returncode}")

In [ ]:
# 11. Results per task x condition, then per condition
try:
    import pandas as pd
    st = json.load(open(f"{STUDY}/state.json"))
    rows = []
    for k, v in st.items():
        if "|" not in k:
            continue
        task, cond = k.split("|")
        us = [v[x] for x in v if x.endswith("_usage")]
        fin = v.get("final_version")
        rows.append({"task": task.split(":")[1], "cond": cond, "finished": v.get("finished", False),
                     "v1_heldout": v.get("v1_heldout_success"), "final": fin,
                     "final_heldout": v.get(f"{fin}_heldout_success") if fin else None,
                     "dev_by_version": [v.get(f"v{i}_dev_success") for i in range(1, 7) if f"v{i}_dev_success" in v],
                     "calls": len(us), "in_tok": sum(u["input"] + u["cache_read"] + u["cache_write"] for u in us),
                     "out_tok": sum(u["output"] for u in us)})
    df = pd.DataFrame(rows).sort_values(["task", "cond"]); display(df)
    done = df[df.finished]
    agg = done.groupby("cond").agg(tasks=("task", "count"), one_shot=("v1_heldout", "sum"), final=("final_heldout", "sum"),
                                   calls=("calls", "sum"), in_tok=("in_tok", "sum"))
    agg["one_shot_rate"] = (agg.one_shot / (5 * agg.tasks)).round(2); agg["final_rate"] = (agg.final / (5 * agg.tasks)).round(2)
    display(agg)
except Exception as e:
    print("results table failed (the data is still saved):", repr(e))

In [ ]:
# 12. Final checkpoint
ckpt.pack(RUN, CK_PATHS)
ckpt.save(RUN, CK_PATHS, "final")
watch_stop.set()
!pkill -f supervise_services.sh
print("RUN4 SESSION DONE")